# Neuronales Netz - Aktivierungsfunktion, Lernrate, Batch-Size, Epochenanzahl

**systematische Einschränkung der Hyperparameterbreiche**

**ki-308** California House Prising

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LinearRegression

from utils.data import load_and_clean_data, get_train_test_split, Standard_Scaler, Min_Max_Scaler
from utils.evaluation import evaluate_predictions, add_result
from utils.plotting import plot_features_vs_target, plot_predicted_vs_actual, plot_residuals, save_fig

plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print(f"TensorFlow Version: {tf.__version__}")
print(f"Numpy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print(f"python Version: {sys.version}")

In [ ]:
def build_model(hidden_layers, activation='relu', learning_rate=0.001):
    """Erstelle ein Sequential-Modell mit gegebener Architektur."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    
    model.add(layers.Dense(1))  # Regression Output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


def train_and_evaluate(model, name, x_train, y_train, x_val, y_val, x_test, y_test,
                       epochs=100, batch_size=32, verbose=0):

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )

    y_train_pred = model.predict(x_train, verbose=0).flatten()
    y_test_pred = model.predict(x_test, verbose=0).flatten()

    result = evaluate_predictions(y_train, y_train_pred, y_test, y_test_pred, name)
    add_result(result)

    return history, result

## Daten laden und transformieren

Da wir sehr verschiedene Intervalle haben sollte eine Skalierung sehr sinnvoll sein, um ansonsten z.B. Sigmoid nicht gut funktionieren kann.

In [ ]:
df = load_and_clean_data()
X_train, X_test, y_train, y_test, feature_names = get_train_test_split(df)

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train, y_train = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")



Beginn mit ReLu: (https://www.datacamp.com/de/tutorial/introduction-to-activation-functions-in-neural-networks?dc_referrer=https%3A%2F%2Fwww.google.com%2F)

- rechnerisch kostengünstig auch bei Anwendung mehrerer Schichten
- gängiste Aktivierungsfunktion

Fixparameter:
- Loss: MSE
- Hidden Layers: 2 mit 64 und 32 units (Datensatz zu klein für tieferes Netz und zu geringe Features für breiteres Netz; abnehmende Größen beugen Overfitting vor [Copilot Promt: welche fixparameter wie layer-, unitanzahl etc. sollte man hier wählen?])
- Metric: mae
- learning_rate: 0.001
- Epochen: 100 (von Florian übernommen)
- Batch-Size: 32 (von Florian übernommen)

In [ ]:
model = keras.Sequential()
model.add(layers.Input(shape=(X_train_min0_max1.shape[1],)))
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(32, activation="relu"))
model.add(layers.Dense(1))  # Regression Output
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)
train_and_evaluate(model, "NN_min0_max1_relu", X_train_min0_max1, y_train_min0_max1, X_val_min0_max1, y_val_min0_max1, X_test_min0_max1, y_test_min0_max1, epochs=100, batch_size=32, verbose=0)


## Beste Kombination aus Aktivierungsfunktion und Skalierung finden

Diese Zeile muss ausgeführt werden, damit später x_train etc. überhaupt definiert ist.

In [ ]:
activations = ['relu', 'tanh', 'sigmoid', 'elu', 'selu', 'leaky_relu']
dataset = [("normal", X_train, y_train, X_val, y_val, X_test, y_test),
           ("standard", X_train_standard, y_train_standard, X_val_standard, y_val_standard, X_test_standard, y_test_standard),
           ("minmax", X_train_min0_max1, y_train_min0_max1, X_val_min0_max1, y_val_min0_max1, X_test_min0_max1, y_test_min0_max1)]

for activation in activations:
    for scale_name, x_train, y_train, x_val, y_val, x_test, y_test in dataset:
        model_name = f"NN_{activation}_{scale_name}"
        model = keras.Sequential()
        model.add(layers.Input(shape=(x_train.shape[1],)))
        model.add(layers.Dense(64, activation=activation))
        model.add(layers.Dense(32, activation=activation))
        model.add(layers.Dense(1))  # Regression Output
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='mse',
            metrics=['mae'],
        )
        train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)        



Wenn man sich nur den Testscore anschaut gewinnt der Tanh + Standardscaler, was logisch ist, da der Tanh daten um die Null herum erwartet und etwas glatter ist und daher nicht so zu overfitting neigt (siehe Relu 6% Abweichung Training und Test). Daher als beste Kombination angesehen

## Lernrate, Batch-size und Epochenanzahl

Weitere Fixparameter sind nun: Tanh und Standardscaler

Vorgehen wie in diesem Artikel beschrieben: https://www.kdnuggets.com/tuning-hyperparameters-in-neural-networks?utm_source=copilot.com

Mit der Lernrate anfangen für bessere Konvergenz -> erstes händisches Gefühl für die richtige Größenordnung

In [ ]:
learning_rate = [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 10.0]

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)    

Aus obigen Ergebnissen in folgendem Bereich genauer suchen: zwischen 0.001 - 0.01

In [ ]:
learning_rate = np.linspace(0.001, 0.01, 50)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Die Auswertung über die Lernraten zeigt kein globales Optimum sondern nur lokale Minima, die für genau dieses Modell: Aktivierungsfkt., Batch-Size, etc. gelten. Daher sollte die Lernrate im Finalen Modell erneut als Hyperparameter getestet werden. Das momentane Optimum liegt bei 0,0054

Test, ob 0,0054 ebenso gut ist wie 0,005408163265306123

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Und was passiert bei einfach nur erneuten Durchläufen des selben Codes?

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Resultate des mehrmaligen Durchlaufes: Abweichung von bis zu 3 % beobachtet. Daher unerhablich welche Verbesserungen bis jetzt durch Hyperparametrisierung erreicht worden sind.
-> Initialgewichte Keras
-> Adam nutzt zufällige Initialisierung und unterschiedliche Ausführungsreihenfolge auf CPU/GPU
-> Reproduzierbarkeit von Floating-Points

Idee: Statistische Untersuchung von Mittelwert und Abweichung bei erneuter Ausführung des selben Modells:

In [ ]:
data = []

for i in range(1,101):

    model_name = f"NN_model_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    history, result = train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  
    data.append(result['R² Test'])

bin_heights, bin_borders = np.histogram(data, bins=30, visible=True)
bin_centers = (bin_borders[:-1] + bin_borders[1:]) / 2

def gaussian(x, A, mu, sigma):
    return A * np.exp(- ((x - mu)**2 / (2 * sigma**2))) 

mean_guess = np.mean(data)
sigma_guess = np.std(data)
max_guess = np.max(bin_heights)

popt, pcov = curve_fit(gaussian, bin_centers, bin_heights, p0=[max_guess, mean_guess, sigma_guess])
fitted_A, fitted_mu, fitted_sigma = popt
fit_variance = fitted_sigma ** 2

print(f"Mittelwert (mu): {fitted_mu:.4f}")
print(f"Standardabweichung (sigma): {fitted_sigma:.4f}")
print(f"Varianz: {fit_variance:.4f}")

plt.hist(data, bins=30, alpha=0.6, color='g', label='R² Test Scores')
x_fit = np.linspace(min(bin_centers), max(bin_centers), 100)
plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', label='Gaussian Fit')
plt.legend()
plt.xlabel('R² Test Score')
plt.ylabel('Häufigkeit')
plt.title('Verteilung der R² Test Scores mit Gaussian Fit')
plt.show()

### Batch-Size

In [ ]:
batch_size = [4, 8, 16, 32, 64, 128, 256, 512]

for batch_size in batch_size:

    model_name = f"NN_BatchSize_{batch_size}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=batch_size, verbose=0) 

### Epochenanzahl

In [ ]:
epochs = [10, 50, 100, 200, 500, 1000, 10000]

for epochs in epochs:

    model_name = f"NN_Epochs_{epochs}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=epochs, batch_size=32, verbose=0) 

In [ ]:
# Modell definieren
model = keras.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0054),
    loss='mse',
    metrics=['mae'],
)

# Listen zum Speichern der Test-Scores
test_mae_per_epoch = []
test_mse_per_epoch = []

# Training über 500 Epochen, aber jede Epoche einzeln
for epoch in range(1, 501):

    model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=1,
        batch_size=32,
        verbose=0,
        shuffle=False
    )

    # Testdaten evaluieren
    mse, mae = model.evaluate(x_test, y_test, verbose=0)
    test_mae_per_epoch.append(mae)
    test_mse_per_epoch.append(mse)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(test_mae_per_epoch, label="Test MAE")
plt.xlabel("Epoche")
plt.ylabel("MAE")
plt.title("Test-MAE pro Epoche (1–500)")
plt.grid(True)
plt.legend()
plt.show()
